### Import Libraries

In [6]:
import os
import re
import sys
import json
import time
import csv
import yaml
import warnings
warnings.filterwarnings('ignore')
import nest_asyncio
nest_asyncio.apply()

In [7]:
from typing import List, Optional
from pydantic import BaseModel, Field
from crewai import Agent, Task, Crew, Process
from crewai import LLM
from crewai_tools import (ScrapeWebsiteTool, WebsiteSearchTool, SerperDevTool, FileReadTool, FileWriterTool, CSVSearchTool, 
                          CodeInterpreterTool, NL2SQLTool, PDFSearchTool)
from crewai.tools import tool, BaseTool
from crewai.knowledge.source.pdf_knowledge_source import PDFKnowledgeSource
from crewai.knowledge.source.csv_knowledge_source import CSVKnowledgeSource
from crewai.knowledge.source.crew_docling_source import CrewDoclingSource

### Loading Tasks and Agents YAML Files

In [8]:
files = {
    'agents': 'config/agents.yaml',
    'tasks': 'config/tasks.yaml'
}

configs = {}
for config_type, file_path in files.items():
    with open(file_path, 'r') as file:
        configs[config_type] = yaml.safe_load(file)

agents_config = configs['agents']
tasks_config = configs['tasks']

### Initialize LLMs

In [9]:
WATSONX_URL = os.environ["WATSONX_URL"] = os.getenv("WATSONX_URL")  
WATSONX_APIKEY = os.environ["WATSONX_APIKEY"] = os.getenv("WATSONX_APIKEY") 
WATSONX_PROJECT_ID = os.environ["WATSONX_PROJECT_ID"] = os.getenv("WATSONX_PROJECT_ID")
WATSONX_MODEL_ID = os.environ["WATSONX_MODEL_ID"] = "watsonx/ibm/granite-3-2-8b-instruct-preview-rc"    # ibm/granite-3-8b-instruct , ibm/granite-3-2-8b-instruct-preview-rc, mistralai/mistral-large

wx_llm = LLM(
    model = WATSONX_MODEL_ID,
    base_url = WATSONX_URL,
    project_id = WATSONX_PROJECT_ID,
	api_key = WATSONX_APIKEY,
    max_tokens = 10000,
    temperature = 0.1
)

### Define Structured Output

In [10]:
from pydantic import BaseModel, Field
from typing import List, Optional

# Resume Screener Agent Output
class ParsedResume(BaseModel):
    candidate_name: str = Field(..., description="The full name of the candidate.")
    primary_skills: List[str] = Field(..., description="The main skills possessed by the candidate.")
    work_experience: List[str] = Field(..., description="Details of past job roles, duration, and impact.")
    education: List[str] = Field(..., description="Educational qualifications of the candidate.")
    notable_achievements: Optional[List[str]] = Field(None, description="Key achievements relevant to job applications.")

# Job Matching Specialist Output
class ResumeMatchEvaluation(BaseModel):
    match_score: int = Field(..., ge=0, le=100, description="The match score indicating how well the candidate fits the job on a scale of 0 to 100.")
    key_strengths: List[str] = Field(..., description="Areas where the candidate strongly aligns with job requirements.")
    gaps: Optional[List[str]] = Field(None, description="Areas where the candidate lacks required skills or experience.")

# Final Structured Output
class ResumeComparisonResult(BaseModel):
    parsed_resume: ParsedResume = Field(..., description="Extracted candidate profile details.")
    match_evaluation: ResumeMatchEvaluation = Field(..., description="Comparison results between resume and job requirements.")

### Define Tools

In [ ]:
from pathlib import Path

job_description_example = "job_description_example.md"
root_folder = Path("..").resolve()  
found_files = list(root_folder.rglob(job_description_example))
jd_sample_file = str(found_files[0])

file_read_tool = FileReadTool(file_path=jd_sample_file)
file_write_tool = FileWriterTool()

from crewai.knowledge.source.pdf_knowledge_source import PDFKnowledgeSource
pdf_source = PDFKnowledgeSource(
    file_paths=["sample.pdf"]
)

### Define Agents

In [12]:
resume_screener = Agent(
        config = agents_config['resume_screener'],
        verbose = True,
        llm = wx_llm,
        allow_delegation = False,
        knowledge_sources = [pdf_source]
)

job_matching_specialist = Agent(
    config = agents_config['job_matching_specialist'],
    verbose = True,
    llm = wx_llm,
    tools = [file_read_tool],
    allow_delegation = True
)

agents_list = [resume_screener, job_matching_specialist]

### Define Tasks

In [13]:
parse_resume_task = Task(
    config = tasks_config['parse_resume_task'],
    agent = resume_screener,
    output_pydantic = ParsedResume
)

match_resume_task = Task(
    config = tasks_config['match_resume_task'],
    agent = job_matching_specialist,
    context = [parse_resume_task],
    output_pydantic = ResumeComparisonResult
)

tasks_list = [parse_resume_task, match_resume_task]

### Run Crew

In [14]:
match_to_proposal_crew = Crew(
    agents = agents_list, 
    tasks = tasks_list, 
    process = Process.sequential,
    verbose = True,
    # process=Process.hierarchical, 
)

In [ ]:
from crewai import Flow
from crewai.flow.flow import listen, start

class ResumeFilteringFlow(Flow):
    @start()
    def fetch_candidates(self):
        # Pull candidates and resumes from the database
        candidates = [
            {
                "resume_path": "sample.pdf",
                "candidate_data": {
                    "candidate_id": 1,
                    "candidate_name": "sample name"
                }
            }
        ]
        return candidates

    @listen(fetch_candidates)
    async def process_resumes(self, candidates):
        scores = await match_to_proposal_crew.kickoff_for_each_async(candidates)
        
        self.state["resume_scores"] = scores
        return scores

    @listen(process_resumes)
    def store_scores(self, scores):
        # Store scores in the database
        return scores

    @listen(process_resumes)
    def filter_candidates(self, scores):
        return [score for score in scores if score['match_evaluation'].match_score >= 80]

    @listen(filter_candidates)
    def return_filtered_results(self, filtered_candidates):
        return filtered_candidates

flow = ResumeFilteringFlow()

In [ ]:
filtered_resumes = flow.kickoff()

In [ ]:
print("Final State:")
print(flow.state)

In [ ]:
print(f"Final Output:\n {filtered_resumes[0].raw}")


In [37]:
import pandas as pd
df_usage_metrics = pd.DataFrame([flow.state["resume_scores"][0].token_usage.dict()])
costs = 0.150 * df_usage_metrics['total_tokens'].sum() / 1_000_000
print(f"Total costs: ${costs:.4f}")
df_usage_metrics

Total costs: $0.0022


,total_tokens,prompt_tokens,cached_prompt_tokens,completion_tokens,successful_requests
0,14941,12185,0,2756,5


In [38]:
flow.plot()

Plot saved as crewai_flow.html


In [40]:
# from IPython.display import IFrame

# IFrame(src='./crewai_flow.html', width='150%', height=600)